# AML - RARE VARIANT POLYGENIC RISK SCORE - TOP GENES 
This script generates a rare variant polygenic risk score file for the top three significant genes identified in the SAIGE-GENE+ set-based analysis. The resulting file contains one column per gene, representing the rare-variant polygenic risk score burden for that gene for each individual. Each predicted deleterious rare variant (missense or loss-of-function; MAF < 1%) present within a gene for a given individual is assigned a weight of 1. The gene-level score is then computed as the sum of all predicted deleterious rare variants present within that gene for each individual.

In [ ]:
# Load packages 
import re
import os
import gc
import glob
import pandas as pd 
from functools import reduce

In [ ]:
# Set bucket environment
my_bucket = os.getenv('WORKSPACE_BUCKET')

In [ ]:
# Remove burden polygenic risk score files 
!rm -rf *prs.tsv

In [ ]:
# Create directory for gene lists 
!rm -rf gene_list
!mkdir -p gene_list 

In [ ]:
# Create directory for burden scores 
!mkdir -p burden_sum_scores

In [ ]:
%%bash 
# Set workspace bucket containing the gene burden sum scores 
my_bucket="gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe"
# Create directory for gene burden sum scores 
mkdir -p burden_scores 
# Loop to download file for the burden sum score for each gene 
for chrom in {1..22}; do
  for gene in ENSG00000119772 ENSG00000168769 ENSG00000161547; do
  mkdir -p burden_scores/burden_scores_chr${chrom}
  gsutil cp "${my_bucket}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chrom}/*${gene}-chr${chrom}.sscore" burden_scores/burden_scores_chr${chrom}
  done
done 

In [ ]:
# Print chromosome directories for burden sum scores 
!ls burden_scores

In [ ]:
# Gene top genes discovered in set-based gene associations 
genes = pd.DataFrame({"gene_id" : ["ENSG00000119772", "ENSG00000168769", "ENSG00000161547"]})

In [ ]:
# ===========================================================================================
# Generate top genes rare variant polygenic risk score file
# ===========================================================================================
# Initialize merged master dataframe
merged_master = None 
# Loop through each chromosome
for chrom in range(1, 23):
    print(f"Processing chromosome {chrom} ....")   
    # Get list of burden score files for the chromosome
    folder_path = f"burden_scores/burden_scores_chr{chrom}"
    files = sorted(glob.glob(os.path.join(folder_path, "*")))

    if not files:
        print(f"No files found for chromosome {chrom}, skipping...")
        continue
    # Initialize master dataframe with FID and IID from the first file
    master = pd.read_csv(files[0], sep="\t")[["#FID", "IID"]]
    # Loop through each file and merge relevant data 
    for filename in files:
        match = re.search(r"(ENSG\d+)", filename)
        if not match:
            continue
        # Extract gene ENSG ID from filename
        ensg_id = match.group(1)

        if ensg_id not in genes["gene_id"].unique():
            continue
        # Read the burden score file
        df = pd.read_csv(filename, sep="\t")
        # Validate that NAMED_ALLELE_DOSAGE_SUM matches SCORE1_SUM
        if not (df["NAMED_ALLELE_DOSAGE_SUM"] == df["SCORE1_SUM"]).all():
            continue
      
        print(f"Processing gene {ensg_id} ....")
        # Keep only relevant columns and rename SCORE1_SUM column
        df = df[["#FID", "IID", "SCORE1_SUM"]].rename(
                columns={"SCORE1_SUM": f"{ensg_id}_SCORE1_SUM"}
            )
        # Merge with master dataframe
        master = pd.merge(master, df, on=["#FID", "IID"], how="outer")
    # Merge chromosome master with overall merged_master
    if merged_master is None:
        merged_master = master
    else:
        merged_master = pd.merge(merged_master, master, on=["#FID", "IID"], how="outer")
    
    print(f"Chromosome {chrom} merge complete. Master shape: {master.shape}")

print("All chromosomes merged.")
print("Final shape:", merged_master.shape)
# Save the final merged master dataframe to a TSV file
merged_master.to_csv(f"top_genes_prs.tsv", sep ='\t', index=False)
# Clean up memory
del master, merged_master
gc.collect()


In [ ]:
# Copy top gene polygenick risk score file to workspace bucket 
!gsutil cp top_genes_prs.tsv {my_bucket}/data/gene_burden_analysis/burden_gene_sets/top_genes/ 